# MotoGP Same Nation Podium Lockouts - Data Preparation

This notebook cleans and standardizes the podium lockouts dataset based on insights from the exploration phase.

## 0. Setup and Data Loading

In [ ]:
# Import necessary libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Configure plot styles
plt.style.use('default')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# Load raw data
raw_data_path = Path("../data/raw/same_nation_podium_lockouts.csv")
df_raw = pd.read_csv(raw_data_path)

print(f"Raw data loaded: {df_raw.shape}")
print(f"Columns: {list(df_raw.columns)}")
df_raw.head()

## 1. Data Quality Assessment

In [ ]:
# Assess data quality
print("=== DATA QUALITY ASSESSMENT ===")
print(f"Total lockout events: {len(df_raw)}")
print(f"Date range: {df_raw['Season'].min()} - {df_raw['Season'].max()}")
print(f"Unique nations: {df_raw['Riders` Nation'].nunique()}")
print(f"Unique tracks: {df_raw['Track'].nunique()}")
print(f"Unique classes: {df_raw['Class'].nunique()}")

print(f"\nMissing values:")
print(df_raw.isnull().sum())

print(f"\nData types:")
print(df_raw.dtypes)

print(f"\nUnique values per column:")
for col in df_raw.columns:
    print(f"{col}: {df_raw[col].nunique()} unique values")

## 2. Column Standardization

In [ ]:
# Create working copy and standardize columns
df = df_raw.copy()

# Standardize column names
column_mapping = {
    'Season': 'season',
    'Track': 'track',
    'Riders` Nation': 'nation',
    'Class': 'class'
}

df = df.rename(columns=column_mapping)

print("Standardized columns:")
print(list(df.columns))
df.head()

## 3. Data Type Corrections

In [ ]:
# Ensure proper data types
df['season'] = df['season'].astype(int)

# Clean text fields
df['track'] = df['track'].astype(str).str.strip()
df['nation'] = df['nation'].astype(str).str.strip().str.upper()
df['class'] = df['class'].astype(str).str.strip()

print("Updated data types:")
print(df.dtypes)

print(f"\nSeason range: {df['season'].min()} - {df['season'].max()}")
print(f"Total seasons with lockouts: {df['season'].nunique()}")

## 4. Nation Code Standardization

In [ ]:
# Standardize nation codes
print("=== NATION CODE STANDARDIZATION ===")

df['nation_clean'] = df['nation'].str.strip().str.upper()

# Show nation distribution
print(f"Nation lockout distribution:")
nation_counts = df['nation_clean'].value_counts()
print(nation_counts)

# Manual corrections for any inconsistencies
nation_corrections = {
    # Add specific corrections if needed
    # Example: 'SPAIN': 'ES', 'ITALY': 'IT'
}

if nation_corrections:
    df['nation_clean'] = df['nation_clean'].replace(nation_corrections)
    print(f"Applied {len(nation_corrections)} nation code corrections")

# Map to full country names for better readability
country_name_mapping = {
    'ES': 'Spain',
    'IT': 'Italy', 
    'GB': 'Great Britain',
    'FR': 'France',
    'DE': 'Germany',
    'AU': 'Australia',
    'US': 'United States',
    'JP': 'Japan',
    'NL': 'Netherlands',
    'CH': 'Switzerland',
    'AT': 'Austria',
    'SE': 'Sweden',
    'FI': 'Finland',
    'BE': 'Belgium',
    'BR': 'Brazil',
    'AR': 'Argentina',
    'ZA': 'South Africa'
}

df['country_name'] = df['nation_clean'].map(country_name_mapping)
df['country_name'] = df['country_name'].fillna(df['nation_clean'])  # Keep original if no mapping

print(f"\nNation distribution with full names:")
nation_name_counts = df['country_name'].value_counts()
print(nation_name_counts)

print(f"Final unique nations: {df['nation_clean'].nunique()}")

## 5. Track Name Standardization

In [ ]:
# Standardize track names
print("=== TRACK NAME STANDARDIZATION ===")
print(f"Original unique tracks: {df['track'].nunique()}")

df['track_clean'] = df['track'].str.strip()

# Show tracks with most lockouts
track_lockouts = df['track_clean'].value_counts()
print(f"\nTop 15 tracks by lockout frequency:")
print(track_lockouts.head(15))

# Manual corrections for track names if needed
track_corrections = {
    # Add specific corrections if found during exploration
    # Example: 'Old track name': 'Standardized track name'
}

if track_corrections:
    df['track_clean'] = df['track_clean'].replace(track_corrections)
    print(f"Applied {len(track_corrections)} track name corrections")

print(f"Final unique tracks: {df['track_clean'].nunique()}")

## 6. Class Standardization and Categorization

In [ ]:
# Standardize racing classes
print("=== CLASS STANDARDIZATION ===")

df['class_clean'] = df['class'].str.strip()

print(f"Class distribution:")
class_counts = df['class_clean'].value_counts()
print(class_counts)

# Create broader class categories
def categorize_class(class_name):
    class_name = str(class_name).upper()
    if 'MOTOGP' in class_name or '500' in class_name:
        return 'Premier Class'
    elif 'MOTO2' in class_name or '250' in class_name:
        return 'Intermediate Class' 
    elif 'MOTO3' in class_name or '125' in class_name:
        return 'Lightweight Class'
    elif 'MOTOE' in class_name:
        return 'Electric Class'
    else:
        return 'Other Class'

df['class_category'] = df['class_clean'].apply(categorize_class)

print(f"\nClass categories:")
print(df['class_category'].value_counts())

# Show lockouts by nation and class
print(f"\nLockouts by nation and class:")
nation_class_cross = pd.crosstab(df['country_name'], df['class_category'], margins=True)
print(nation_class_cross)

## 7. Temporal Analysis and Grouping

In [ ]:
# Add temporal groupings
print("=== TEMPORAL ANALYSIS ===")

# Create decade grouping
df['decade'] = (df['season'] // 10) * 10

# Create era grouping
def categorize_era(season):
    if season < 1980:
        return 'Classic Era (pre-1980)'
    elif season < 2000:
        return 'Modern Era (1980-1999)'
    elif season < 2020:
        return 'Contemporary Era (2000-2019)'
    else:
        return 'Current Era (2020+)'

df['era'] = df['season'].apply(categorize_era)

print(f"Lockouts by decade:")
decade_counts = df['decade'].value_counts().sort_index()
print(decade_counts)

print(f"\nLockouts by era:")
era_counts = df['era'].value_counts()
print(era_counts)

# Lockouts by nation and decade
print(f"\nTop 3 nations by decade:")
for decade in sorted(df['decade'].unique()):
    decade_data = df[df['decade'] == decade]
    if len(decade_data) > 0:
        top_nations = decade_data['country_name'].value_counts().head(3)
        print(f"{decade}s: {list(top_nations.items())}")

## 8. Continental Mapping

In [ ]:
# Add continental mapping
print("=== CONTINENTAL MAPPING ===")

# Continental mapping consistent with other datasets
continent_mapping = {
    'ES': 'Europe', 'IT': 'Europe', 'GB': 'Europe', 'FR': 'Europe', 'DE': 'Europe',
    'NL': 'Europe', 'CH': 'Europe', 'AT': 'Europe', 'SE': 'Europe', 'FI': 'Europe',
    'BE': 'Europe', 'PT': 'Europe', 'IE': 'Europe', 'NO': 'Europe', 'DK': 'Europe',
    
    'JP': 'Asia', 'MY': 'Asia', 'TH': 'Asia', 'CN': 'Asia', 'IN': 'Asia', 'ID': 'Asia',
    
    'AU': 'Oceania', 'NZ': 'Oceania',
    
    'US': 'North America', 'CA': 'North America', 'MX': 'North America',
    
    'BR': 'South America', 'AR': 'South America', 'VE': 'South America',
    
    'ZA': 'Africa', 'EG': 'Africa', 'MA': 'Africa'
}

df['continent'] = df['nation_clean'].map(continent_mapping)
df['continent'] = df['continent'].fillna('Other')

print(f"Continental distribution of lockouts:")
continent_counts = df['continent'].value_counts()
print(continent_counts)

# Check for unmapped countries
unmapped = df[df['continent'] == 'Other']['nation_clean'].unique()
if len(unmapped) > 0:
    print(f"\nUnmapped nations: {unmapped}")
else:
    print("\nAll nations successfully mapped to continents")

print(f"\nLockouts by continent and era:")
continent_era_cross = pd.crosstab(df['continent'], df['era'], margins=True)
print(continent_era_cross)

## 9. Dominance Metrics

In [ ]:
# Calculate dominance metrics
print("=== DOMINANCE METRICS ===")

# Nation dominance metrics
nation_metrics = df.groupby('nation_clean').agg({
    'season': ['count', 'min', 'max', 'nunique'],
    'track_clean': 'nunique',
    'class_clean': 'nunique'
})

nation_metrics.columns = ['total_lockouts', 'first_lockout', 'last_lockout', 'seasons_active', 'tracks_dominated', 'classes_dominated']
nation_metrics['lockout_span'] = nation_metrics['last_lockout'] - nation_metrics['first_lockout']
nation_metrics['avg_lockouts_per_active_season'] = nation_metrics['total_lockouts'] / nation_metrics['seasons_active']

# Add country names and continents
nation_metrics['country_name'] = nation_metrics.index.to_series().map(country_name_mapping)
nation_metrics['country_name'] = nation_metrics['country_name'].fillna(nation_metrics.index)
nation_metrics['continent'] = nation_metrics.index.to_series().map(continent_mapping)
nation_metrics['continent'] = nation_metrics['continent'].fillna('Other')

# Sort by total lockouts
nation_metrics = nation_metrics.sort_values('total_lockouts', ascending=False)

print("Nation dominance metrics:")
print(nation_metrics.head(10))

# Track dominance (tracks that frequently see lockouts)
track_dominance = df.groupby('track_clean').agg({
    'season': ['count', 'nunique'],
    'nation_clean': 'nunique',
    'class_clean': 'nunique'
})

track_dominance.columns = ['total_lockouts', 'seasons_with_lockouts', 'nations_involved', 'classes_involved']
track_dominance = track_dominance.sort_values('total_lockouts', ascending=False)

print(f"\nTop 10 tracks by lockout frequency:")
print(track_dominance.head(10))

## 10. Data Validation

In [ ]:
# Comprehensive data validation
print("=== DATA VALIDATION ===")

# Check season ranges
valid_seasons = (df['season'] >= 1949) & (df['season'] <= 2025)
invalid_seasons = (~valid_seasons).sum()
if invalid_seasons > 0:
    print(f"Warning: {invalid_seasons} lockouts with invalid seasons")
else:
    print("✓ All seasons are within expected range")

# Check for missing data in required fields
required_fields = ['season', 'track_clean', 'nation_clean', 'class_clean']
for field in required_fields:
    missing = df[field].isnull().sum()
    if missing > 0:
        print(f"Warning: {missing} missing values in {field}")
    else:
        print(f"✓ No missing values in {field}")

# Check for logical consistency
# Each lockout should represent a unique combination of season, track, and class
duplicates = df.duplicated(subset=['season', 'track_clean', 'class_clean']).sum()
if duplicates > 0:
    print(f"Note: {duplicates} duplicate lockout combinations (may be valid)")
    # Show examples
    duplicate_examples = df[df.duplicated(subset=['season', 'track_clean', 'class_clean'], keep=False)]
    print("Examples:")
    print(duplicate_examples[['season', 'track_clean', 'class_clean', 'nation_clean']].head())
else:
    print("✓ No duplicate lockout combinations")

print(f"\n=== VALIDATION SUMMARY ===")
print(f"Total lockout events after preparation: {len(df)}")
print(f"Season range: {df['season'].min()} - {df['season'].max()}")
print(f"Unique nations: {df['nation_clean'].nunique()}")
print(f"Unique tracks: {df['track_clean'].nunique()}")
print(f"Unique classes: {df['class_clean'].nunique()}")
print(f"Unique seasons: {df['season'].nunique()}")

## 11. Final Dataset Structure

In [ ]:
# Define final column order
final_columns = [
    # Core data
    'season', 'track', 'track_clean', 'nation', 'nation_clean', 'country_name', 'continent',
    'class', 'class_clean', 'class_category',
    # Temporal groupings
    'decade', 'era'
]

# Create final dataset
df_final = df[final_columns].copy()

print("=== FINAL PREPARED DATASET ===")
print(f"Shape: {df_final.shape}")
print(f"Columns: {list(df_final.columns)}")

print(f"\nSample of prepared data:")
print(df_final.head(15))

print(f"\nData types:")
print(df_final.dtypes)

print(f"\nMissing values:")
print(df_final.isnull().sum())

## 12. Save Prepared Data

In [ ]:
# Create prepared data directory if it doesn't exist
prepared_data_path = Path("../../data/interim/")
prepared_data_path.mkdir(parents=True, exist_ok=True)

# Save prepared dataset
output_file = prepared_data_path / "podium_lockouts_prepared.csv"
df_final.to_csv(output_file, index=False)

print(f"Prepared dataset saved to: {output_file}")
print(f"File size: {output_file.stat().st_size:,} bytes")

# Also save nation metrics for reference
nation_metrics_file = prepared_data_path / "nation_dominance_metrics.csv"
nation_metrics.to_csv(nation_metrics_file)
print(f"Nation metrics saved to: {nation_metrics_file}")

# Save track dominance metrics
track_metrics_file = prepared_data_path / "track_lockout_metrics.csv"
track_dominance.to_csv(track_metrics_file)
print(f"Track metrics saved to: {track_metrics_file}")

# Verification
df_verification = pd.read_csv(output_file)
print(f"\nVerification - loaded shape: {df_verification.shape}")
print("✓ Files saved and verified successfully")

## 13. Preparation Summary

In [ ]:
print("=== PREPARATION SUMMARY ===")
print("\n✅ COMPLETED TASKS:")
print("1. Column name standardization")
print("2. Data type corrections and validation")
print("3. Nation code standardization and mapping to full names")
print("4. Track name cleaning and standardization")
print("5. Class standardization and categorization")
print("6. Temporal grouping (decades, eras)")
print("7. Continental mapping for geographical analysis")
print("8. Dominance metrics calculation")
print("9. Data validation and consistency checks")
print("10. Prepared dataset and metrics saved")

print("\n📊 DATASET IMPROVEMENTS:")
print(f"- Standardized nation codes with full country names")
print(f"- Added comprehensive continental mapping")
print(f"- Created class categories for broader analysis")
print(f"- Added temporal groupings for era-based analysis")
print(f"- Generated nation and track dominance metrics")

print("\n🏆 NATIONAL DOMINANCE:")
top_nations = df_final['country_name'].value_counts().head(5)
for nation, count in top_nations.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {nation}: {count} lockouts ({percentage:.1f}%)")

print("\n🏁 CLASS DISTRIBUTION:")
class_dist = df_final['class_category'].value_counts()
for category, count in class_dist.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {category}: {count} lockouts ({percentage:.1f}%)")

print("\n📅 TEMPORAL PATTERNS:")
era_dist = df_final['era'].value_counts()
for era, count in era_dist.items():
    percentage = (count / len(df_final)) * 100
    print(f"- {era}: {count} lockouts ({percentage:.1f}%)")

print("\n🔍 KEY INSIGHTS:")
most_dominant_nation = df_final['country_name'].value_counts().index[0]
most_dominant_count = df_final['country_name'].value_counts().iloc[0]
print(f"- Most dominant nation: {most_dominant_nation} with {most_dominant_count} lockouts")

most_lockout_prone_track = df_final['track_clean'].value_counts().index[0]
most_lockout_track_count = df_final['track_clean'].value_counts().iloc[0]
print(f"- Most lockout-prone track: {most_lockout_prone_track} with {most_lockout_track_count} lockouts")

peak_decade = df_final['decade'].value_counts().index[0]
peak_decade_count = df_final['decade'].value_counts().iloc[0]
print(f"- Peak decade for lockouts: {peak_decade}s with {peak_decade_count} lockouts")

print("\n➡️  READY FOR ANALYSIS PHASE")
print("The prepared podium lockouts dataset is now ready for national dominance analysis.")